In [0]:

# discover_entities()
# Find all files in landing zone

# generate entity_config
# Format-agnostic reader - works for any format (csv, json, parquet).

# metadata

# Materialize bronze.<entity> tables
# Persists data in bronze table to each identity accordingly.


In [0]:
def discover_entities(base_path):
    """
    Scan landing zone and discover all entities.
    
    Args:
        base_path: Path to synthea directory
        
    Returns:
        List of entity names (e.g., ["allergies", "patients"])
    """
    # 1. List all items in base_path
    try:
        files = dbutils.fs.ls(base_path)
    except Exception as e: 
        raise ValueError(f"Cannot access landing zone: {e}")

    # 2. Filter to only directories and build list
    entity_names = []  # ← Create list to collect entities
    
    for file in files:
        # Loop through subdirectories only
        if not file.isDir():
            continue
        
        # Extract entity names
        entity_name = file.name.rstrip("/")
        entity_names.append(entity_name)  # ← Add to list
    
    # 3. Return the list
    return entity_names

In [0]:
# Define paths
project_catalog = "healthcare_dev"
landing_schema = "00_landing"
bronze_schema = "01_bronze"
raw_volume = "raw_landing_volume"

# Define locations
base_path = f"/Volumes/{project_catalog}/{landing_schema}/{raw_volume}/synthea/"
checkpoint_base = f"/Volumes/{project_catalog}/{bronze_schema}/checkpoints/"

# Discover all entities
entities = discover_entities(base_path)
print(f"\n🔍 Discovered {len(entities)} entities:")
for entity in entities:
    print(f"  - {entity}")

# Now let's see what the full path looks like for each entity
print(f"\n📁 Full CSV paths:")
for entity in entities:
    csv_path = f"{base_path}{entity}/{entity}.csv"
    print(f"  - {entity}: {csv_path}")

In [0]:
entity_config = { 
    "source_path": "",
    "entity_name": "", 
    "format": "", 
    "checkpoint_path": "",
    "schema_path": ""
}

In [0]:
def ingest_entity(entity_config):
    """
    Summary:
    Apply Auto Loader to handle schema evolution and inference, readStream.

    Args:
     - source_path: where the entity is located.
     - entity_name: name of the entity.
     - format: csv, parquet, json..
        
    Returns:
      - Returns a confirmation that it read the csvs with auto loader for each entity.
        
    Raises:
      - File not found, entity config not in acceptable format.
       
    """

In [0]:
def bronze_ingestion():
    # entities = discover_entities(base_path)
    # for entity in entities: 
        # ingest_entity (entity_config)
    # write to bronze_<entity>
    #writeStream -> auto loader write syntax toTable(bronze)


## Find all files in landing zone

In [0]:
# Define paths
project_catalog = "healthcare_dev"
landing_schema = "00_landing"
bronze_schema = "01_bronze"
raw_volume = "raw_landing_volume"


# Define locations
base_path = f"/Volumes/{project_catalog}/{landing_schema}/{raw_volume}/synthea/"
checkpoint_base = f"/Volumes/{project_catalog}/{bronze_schema}/checkpoints/"
schema_path = f"/{checkpoint_path}_schema"


# List all files
try:
    files = dbutils.fs.ls(base_path)
except Exception as e: 
    raise ValueError(f"Cannot access landing zone: {e}")


for file in files:
    # Extract entity names
    entity_name = file.name.rstrip("/")
    
    # Loop through subdirectories only
    if not file.isDir:
        continue

    # Auto loader for incremental file ingestion
    df_bronze = spark.readStream \ 
                .format("cloudFiles") \
                .option("cloudFiles.format", ".csv") \ 
                .option("cloudFiles.schemaLocation", "")
                .option("header", "true") \ 
                .option("rescuedDataColumn", "_rescued_data") \
                .load(landing_path)

    # Build entity writer
    # direct it to tables 01_bronze.bronze_<entity>


In [0]:
# What's actually in the landing zone?
files = dbutils.fs.ls(landing_path)

print("Files/folders found:")
for f in files:
    print(f"  {'[DIR] ' if f.isDir() else '[FILE]'} {f.name}")

In [0]:
# Define paths
project_catalog = "healthcare_dev"
landing_schema = "00_landing"
bronze_schema = "01_bronze"
raw_volume = "raw_landing_volume"


# Define locations
base_path = f"/Volumes/{project_catalog}/{landing_schema}/{raw_volume}/synthea/"
checkpoint_base = f"/Volumes/{project_catalog}/{bronze_schema}/checkpoints/"
schema_path = f"/{checkpoint_base}_schema"


# Pick ONE entity to test with
test_entity = "patients"

# Questions to answer:
# 1. What's the path to the CSV file?
test_csv_path = f"{base_path}{test_entity}/{test_entity}.csv"
print(f"CSV path: {test_csv_path}")

# 2. Does this file exist?
try:
    file_info = dbutils.fs.ls(test_csv_path)
    print(f"✅ File exists! Size: {file_info[0].size} bytes")
except:
    print("❌ File not found - check the path!")



print(f"Checkpoint: {checkpoint_path}")
print(f"Schema: {schema_path}")
checkpoint_path = f"{checkpoint_base}{test_entity}/"